# Caption + ethogram-classify octopus clips with Qwen3-VL-30B (vLLM / Colab GPU)

For every clip in **`octopus_clips_verified/`**, writes back into **`octopus_clips_verified.json`**:
- a one-sentence **`caption`**, and
- an **`ethogram_label`** chosen from `ethogram_list.json` (or `octopus not present`).

Based on `exp23_caption_colab.ipynb`, wired to the current pipeline (index JSON is the source
of truth). **Resumable** — skips entries that already have a `caption`; saves after every clip.

Uses **Qwen3-VL-30B-A3B-Instruct (AWQ Int4)** via **vLLM** (needs an A100-40GB GPU).

> **Runtime → Change runtime type → A100 GPU.** First model load downloads ~17 GB.

## 1. Install dependencies

In [ ]:
# Remove Colab's cu128 torch FIRST so vLLM installs the torch it was built against
# (else `from vllm import LLM` fails with libcudart.so.13 not found). See exp23 for the why.
!pip uninstall -y -q torch torchvision torchaudio torchao 2>/dev/null
!pip install -q -U vllm qwen-vl-utils
!apt-get -qq install -y ffmpeg >/dev/null

print("Installed.")
print("NOW: Runtime -> Restart session, then run from the CONFIG cell (skip this install cell).")

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

## 2. Config

In [ ]:
from pathlib import Path

MODEL        = "QuantTrio/Qwen3-VL-30B-A3B-Instruct-AWQ"   # AWQ Int4 30B-A3B MoE (~17 GB)

FRAME_FPS     = 0.5     # 1 frame / 2s -> ~10 frames per 20s clip
MAX_TOKENS    = 220
MAX_MODEL_LEN = 8192
GPU_MEM_UTIL  = 0.92
IMAGE_LIMIT   = 16

INDEX_JSON    = Path("octopus_clips_verified.json")   # clip index (we add caption + ethogram_label)
CLIPS_ROOT    = Path("octopus_clips_verified")        # the clips folder (nested date/segment)
ETHOGRAM_PATH = Path("ethogram_list.json")            # behavior list to classify into
CAPTION_KEY   = "caption"
ETHOGRAM_KEY  = "ethogram_label"
ABSENT        = "octopus not present"                 # sentinel when no octopus in the clip

## 3. Get the data into Colab

Put `octopus_clips_verified.zip` (the clips folder), `octopus_clips_verified.json`, and
`ethogram_list.json` in **`MyDrive/GSOC-Catrobat/`**, then run the Drive cell.

In [ ]:
# === Option B: Google Drive  (recommended) ===
from google.colab import drive
drive.mount("/content/drive")

import zipfile, shutil
DRIVE_ROOT = Path("/content/drive/MyDrive/GSOC-Catrobat")

with zipfile.ZipFile(DRIVE_ROOT / "octopus_clips_verified.zip") as z:
    z.extractall(".")                                   # creates ./octopus_clips_verified/
shutil.copy(DRIVE_ROOT / "octopus_clips_verified.json", INDEX_JSON)
shutil.copy(DRIVE_ROOT / "ethogram_list.json", ETHOGRAM_PATH)

print(len(list(CLIPS_ROOT.rglob("*.mp4"))), "clips |", INDEX_JSON, "|", ETHOGRAM_PATH)

In [ ]:
# # === Option A: Zip upload  (SKIP if you used Drive) ===
# import zipfile, shutil
# from google.colab import files
# up = files.upload()   # octopus_clips_verified.zip + octopus_clips_verified.json + ethogram_list.json
# for name in up:
#     if name.endswith(".zip"):
#         with zipfile.ZipFile(name) as z: z.extractall(".")
#     elif name.endswith(".json") and "ethogram" in name: shutil.move(name, ETHOGRAM_PATH)
#     elif name.endswith(".json"): shutil.move(name, INDEX_JSON)
# print(len(list(CLIPS_ROOT.rglob("*.mp4"))), "clips ready")

## 4. Load the model

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoProcessor

print(f"Loading {MODEL} with vLLM ...  (first run downloads ~17 GB)")
llm = LLM(
    model=MODEL, max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=GPU_MEM_UTIL,
    limit_mm_per_prompt={"image": IMAGE_LIMIT}, dtype="auto", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
sampling_params = SamplingParams(temperature=0.0, max_tokens=MAX_TOKENS)
print("Model ready (vLLM).")

## 5. Prompt + ethogram parsing

Passes the ethogram labels + descriptions and asks Qwen for a caption **and** one label from
the list — or `octopus not present` for both if the octopus isn't visible in any frame.

In [ ]:
import json

ethogram  = json.load(open(ETHOGRAM_PATH))
behaviors = ethogram["behaviors"]
labels    = [b["label"] for b in behaviors]
valid_set = set(labels)
label_block = "\n".join(f"- {b['label']}: {b['description']}" for b in behaviors)

def build_prompt() -> str:
    return (
        "These frames are sampled in order from a single short aquarium security-camera clip. "
        "The subject is Nity, an octopus (Octopus vulgaris). "
        "Octopuses change color, extend arms, hide in dens, manipulate objects, and interact with humans.\n\n"
        "First decide whether the octopus appears in ANY frame of the clip.\n"
        "- If the octopus is NOT visible in any frame, respond with EXACTLY:\n"
        "  CAPTION: octopus not present\n"
        "  ETHOGRAM: octopus not present\n"
        "- If the octopus IS visible, write ONE caption describing what it does across the whole clip "
        "(movement, posture, arm position, color, and anything it touches or interacts with), then "
        "pick the SINGLE best-matching behavior from this ethogram and copy its label verbatim:\n"
        f"{label_block}\n\n"
        "Respond in EXACTLY this format, nothing else:\n"
        "CAPTION: <one sentence for the whole clip>\n"
        "ETHOGRAM: <one label copied verbatim from the list above>"
    )

# keyword fallback (same mapping as exp21/exp23) for when the model's label is not verbatim
_KW = [
    (["crawl", "walking on arms", "moving across"], "Crawling"),
    (["swim", "jet", "propel", "water column"], "Swimming / jetting"),
    (["arm walk", "two arm", "bipedal"], "Arm walking"),
    (["hunt", "stalk", "pursuit", "chasing"], "Hunting"),
    (["captur", "pounce", "grab", "catch", "seiz"], "Capturing prey"),
    (["eat", "feeding", "consuming", "tearing food", "food"], "Manipulating food"),
    (["entering den", "into den", "into shelter", "retreating into"], "Entering den"),
    (["exiting den", "emerging", "leaving den", "coming out"], "Exiting den"),
    (["rearrang", "piling", "moving shells", "moving rocks", "den entrance"], "Rearranging den"),
    (["extend", "probing", "reaching out", "arm out", "tentacle out"], "Arm extension / probing"),
    (["manipulat", "picking up", "holding object", "playing with"], "Object manipulation"),
    (["above water", "out of water", "water surface"], "Reaching out of water"),
    (["human", "person", "hand", "researcher", "respond"], "Responding to human"),
    (["joystick", "toy", "enrichment", "device", "screen"], "Enrichment interaction"),
    (["color", "colour", "blanch", "darken", "chromatophore", "texture", "camouflage"], "Color / texture change"),
    (["ink", "cloud"], "Ink release"),
    (["hid", "flatten", "conceal", "press"], "Hiding / flattening"),
    (["stationary", "resting", "motionless", "still", "not moving", "sitting in den", "inside den"], "Stationary in den"),
    (["open area", "tank floor"], "Stationary in open"),
]
def match_ethogram(text: str) -> str:
    t = text.lower()
    for kws, label in _KW:
        if any(k in t for k in kws): return label
    return "unknown"

def parse_response(text: str):
    """Return (caption, ethogram_label); normalize to ABSENT when the octopus isn't present."""
    caption, etho = "", None
    for line in text.splitlines():
        s = line.strip()
        if s.upper().startswith("CAPTION:"):
            caption = s[len("CAPTION:"):].strip().strip("'\"")
        elif s.upper().startswith("ETHOGRAM:"):
            raw = s[len("ETHOGRAM:"):].strip().strip("'\"")
            rl = raw.lower()
            for label in valid_set:
                if label.lower() == rl or label.lower() in rl or rl in label.lower():
                    etho = label; break
            else:
                etho = raw or None
    if not caption:
        caption = text.strip().strip("'\"")
    if "not present" in caption.lower() or "not visible" in caption.lower() or \
       (etho and "not present" in etho.lower()):
        return ABSENT, ABSENT
    if not etho or etho not in valid_set:        # invalid/near-miss label -> keyword fallback
        etho = match_ethogram(f"{caption} {etho or ''}")
    return caption, etho

print(f"{len(labels)} ethogram labels loaded.")

## 6. Frame sampling + inference helpers

In [ ]:
import subprocess, tempfile
from qwen_vl_utils import process_vision_info

MAX_PIXELS = 512 * 512

def resolve_clip(entry) -> Path:
    rel = entry["clip_path"].split("octopus_clips_verified/", 1)[-1]   # <date>/<seg>/<name>.mp4
    return CLIPS_ROOT / rel

def extract_frames(clip_path: Path, tmpdir: str) -> list:
    pattern = str(Path(tmpdir) / "f_%03d.jpg")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(clip_path),
                    "-vf", f"fps={FRAME_FPS}", "-q:v", "2", pattern], check=True)
    return sorted(str(p) for p in Path(tmpdir).glob("f_*.jpg"))

def caption_clip(frame_paths: list, prompt: str) -> str:
    content = [{"type": "image", "image": p, "max_pixels": MAX_PIXELS} for p in frame_paths]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    out = llm.generate({"prompt": text, "multi_modal_data": {"image": image_inputs}},
                       sampling_params=sampling_params, use_tqdm=False)
    return out[0].outputs[0].text.strip()

## 7. Run captioning + classification (resumable)

Writes `caption` and `ethogram_label` onto each entry in `octopus_clips_verified.json`, saving
after every clip. Re-running skips entries that already have a `caption`.

In [ ]:
import json
from datetime import datetime
from collections import Counter

prompt = build_prompt()
index  = json.load(open(INDEX_JSON))
clips  = index["clips"]

todo = [c for c in clips if not c.get(CAPTION_KEY)]
print(f"{len(clips)} clips total, {len(todo)} to caption\n" + "-" * 60)

done_n = 0
for i, entry in enumerate(todo):
    clip_path = resolve_clip(entry)
    print(f"[{i+1}/{len(todo)}] {entry['clip_path']}", flush=True)
    if not clip_path.exists():
        print(f"  ! missing file: {clip_path}"); continue
    with tempfile.TemporaryDirectory() as tmp:
        try:
            frames = extract_frames(clip_path, tmp)
            if not frames:
                print("  no frames"); continue
            raw = caption_clip(frames, prompt)
        except Exception as e:
            print(f"  failed: {e}"); continue

    caption, etho = parse_response(raw)
    entry[CAPTION_KEY]    = caption
    entry[ETHOGRAM_KEY]   = etho
    entry["captioned_at"]  = datetime.now().isoformat(timespec="seconds")
    entry["caption_model"] = MODEL.split("/")[-1]
    print(f"  caption : {caption}")
    print(f"  ethogram: {etho}", flush=True)

    with open(INDEX_JSON, "w") as f:        # save after every clip -> resumable
        json.dump(index, f, indent=2)
    done_n += 1

print("-" * 60 + f"\nDone. captioned {done_n} clips this run.")
print(f"with caption now: {sum(1 for c in clips if c.get(CAPTION_KEY))}/{len(clips)}")
print("\nEthogram distribution:")
for label, n in Counter(c.get(ETHOGRAM_KEY) for c in clips if c.get(ETHOGRAM_KEY)).most_common():
    print(f"  {n:3d}  {label}")

## 8. Save the JSON back

Copies the updated `octopus_clips_verified.json` back to Drive. Then drop it into
`data/octopus_clips_verified.json` in the repo.

In [ ]:
import shutil
if Path("/content/drive/MyDrive").exists():
    shutil.copy(INDEX_JSON, DRIVE_ROOT / INDEX_JSON.name)
    print(f"Copied back to Drive: {DRIVE_ROOT / INDEX_JSON.name}")
else:
    from google.colab import files
    files.download(str(INDEX_JSON))